# Warehouse

Confirmation checks for the warehouse build. Jobs live in `spark/jobs/`.

## Spark session

In [1]:
# confirm connection with Spark
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("warehouse")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    # pinned so the executors have a fixed address to call back on
    .config("spark.driver.port", "7078")
    .config("spark.blockManager.port", "7079")
    # without a cap this session holds every core
    .config("spark.cores.max", 2)
    .config("spark.executor.memory", "1g")
    # one shared catalog on the mount, so tables created by spark-submit
    # resolve here too
    .config("spark.sql.warehouse.dir", "/opt/data/warehouse")
    .config(
        "javax.jdo.option.ConnectionURL",
        "jdbc:derby:;databaseName=/opt/data/metastore_db;create=true",
    )
    .enableHiveSupport()
    .getOrCreate()
)

print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)
print("Application ID:", spark.sparkContext.applicationId)

spark.range(5).selectExpr("id", "id * id AS squared").show()

Spark version : 3.5.0
Master        : spark://spark-master:7077
Application ID: app-20260802230200-0022
+---+-------+
| id|squared|
+---+-------+
|  0|      0|
|  1|      1|
|  2|      4|
|  3|      9|
|  4|     16|
+---+-------+



In [2]:
# confirm the data mount
from pathlib import Path

RAW = Path("/opt/data/raw")

print(f"raw root : {RAW}")
print(f"exists   : {RAW.is_dir()}")

for year in sorted(p for p in RAW.iterdir() if p.is_dir()):
    csvs = sorted(year.glob("*.csv"))
    print(f"{year.name}  {len(csvs):2d} csv")

raw root : /opt/data/raw
exists   : True
2019   4 csv
2020  12 csv
2021  12 csv
2022  11 csv
2023  12 csv
2024   1 csv
2025  12 csv


In [3]:
# confirm the executors can read off the mount, not just the driver
sample = spark.read.csv(
    "/opt/data/raw/2019/2019-Q1.csv", header=True, inferSchema=False
)

print(f"rows    : {sample.count():,}")
print(f"columns : {len(sample.columns)}")
sample.show(3, truncate=False)

rows    : 189,063
columns : 10
+-------+--------------+----------------+----------------+------------------------+--------------+----------------+-----------------------------------+-------+-------------+
|Trip Id|Trip  Duration|Start Station Id|Start Time      |Start Station Name      |End Station Id|End Time        |End Station Name                   |Bike Id|User Type    |
+-------+--------------+----------------+----------------+------------------------+--------------+----------------+-----------------------------------+-------+-------------+
|4581278|1547          |7021            |01/01/2019 00:08|Bay St / Albert St      |7233          |01/01/2019 00:33|King / Cowan Ave - SMART           |1296   |Annual Member|
|4581279|1112          |7160            |01/01/2019 00:10|King St W / Tecumseth St|7051          |01/01/2019 00:29|Wellesley St E / Yonge St (Green P)|2947   |Annual Member|
|4581280|589           |7055            |01/01/2019 00:15|Jarvis St / Carlton St  |7013          |0

## Stage table

Built by [`02_stage_table.py`](../jobs/02_stage_table.py). Empty until the
extract populates the `source_year` partitions.

In [4]:
# confirm stage table creation
WAREHOUSE = "/opt/data/warehouse"
STAGE_TRIPS_PATH = f"{WAREHOUSE}/stage_trips"

path = Path(STAGE_TRIPS_PATH)
partitions = sorted(p.name for p in path.glob("source_year=*"))
data_files = sorted(path.glob("**/*.parquet"))

print(f"path       : {path}")
print(f"exists     : {path.is_dir()}")
print(f"partitions : {', '.join(partitions) or '-'}")
print(f"data files : {len(data_files)}")

path       : /opt/data/warehouse/stage_trips
exists     : True
partitions : source_year=2019, source_year=2020, source_year=2021, source_year=2022, source_year=2023
data files : 45


In [5]:
# confirm the schema: 11 columns, all string
# read via the catalog - at zero rows there are no parquet files to infer from
stage = spark.table("stage_trips")

stage.printSchema()
print(f"columns : {len(stage.columns)}")
print(f"rows    : {stage.count():,}")

root
 |-- trip_id: string (nullable = true)
 |-- trip_duration: string (nullable = true)
 |-- start_time: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- end_time: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- bike_id: string (nullable = true)
 |-- user_type: string (nullable = true)
 |-- model: string (nullable = true)
 |-- source_year: string (nullable = true)

columns : 12
rows    : 18,939,388


In [6]:
# confirm the columns and their order
REFERENCE_COLUMNS = [
    "trip_id",
    "trip_duration",
    "start_time",
    "start_station_id",
    "start_station_name",
    "end_time",
    "end_station_id",
    "end_station_name",
    "bike_id",
    "user_type",
    "model",
]

actual = [f.name for f in stage.schema.fields if f.name != "source_year"]
non_string = [
    f.name
    for f in stage.schema.fields
    if f.name != "source_year" and f.dataType.simpleString() != "string"
]

print(f"missing    : {[c for c in REFERENCE_COLUMNS if c not in actual] or '-'}")
print(f"unexpected : {[c for c in actual if c not in REFERENCE_COLUMNS] or '-'}")
print(f"order      : {'ok' if actual == REFERENCE_COLUMNS else 'differs'}")
print(f"non-string : {non_string or '-'}")

missing    : -
unexpected : -
order      : ok
non-string : -


## Extract

Built by [`03_extract.py`](../jobs/03_extract.py). Raw csv -> stage table,
one partition per source year.

In [7]:
# confirm partitions and row counts per year
stage = spark.table("stage_trips")

stage.groupBy("source_year").count().orderBy("source_year").show()
print(f"total rows : {stage.count():,}")

+-----------+-------+
|source_year|  count|
+-----------+-------+
|       2019|2439517|
|       2020|2911308|
|       2021|3575182|
|       2022|4300240|
|       2023|5713141|
+-----------+-------+

total rows : 18,939,388


In [8]:
# confirm stage rows match the raw csv line count
from pathlib import Path

for year in [2019, 2020, 2021, 2022, 2023]:
    raw = 0
    for csv in sorted(Path(f"/opt/data/raw/{year}").glob("*.csv")):
        with open(csv, encoding="utf-8", errors="replace") as fh:
            raw += sum(1 for _ in fh) - 1  # minus header
    staged = stage.filter(stage.source_year == str(year)).count()
    flag = "ok" if raw == staged else f"MISMATCH ({raw - staged:+,})"
    print(f"{year}  raw {raw:>9,}  staged {staged:>9,}  {flag}")

2019  raw 2,439,517  staged 2,439,517  ok
2020  raw 2,911,308  staged 2,911,308  ok
2021  raw 3,575,182  staged 3,575,182  ok
2022  raw 4,300,240  staged 4,300,240  ok
2023  raw 5,713,141  staged 5,713,141  ok


In [9]:
# confirm no column arrived empty, which would mean a header mismatch
from pyspark.sql import functions as F

nulls = stage.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in stage.columns]
).collect()[0].asDict()

total = stage.count()
for col, n in nulls.items():
    print(f"{col:20s} {n:>9,} null  {n / total:6.1%}")

trip_id                      0 null    0.0%
trip_duration               16 null    0.0%
start_time                   0 null    0.0%
start_station_id             0 null    0.0%
start_station_name           0 null    0.0%
end_time                     0 null    0.0%
end_station_id               0 null    0.0%
end_station_name             0 null    0.0%
bike_id                      0 null    0.0%
user_type                  249 null    0.0%
model                18,939,388 null  100.0%
source_year                  0 null    0.0%


In [10]:
# eyeball a few staged rows
stage.show(5, truncate=False)

+--------+-------------+----------------+----------------+-----------------------------------+----------------+--------------+----------------------------+-------+-------------+-----+-----------+
|trip_id |trip_duration|start_time      |start_station_id|start_station_name                 |end_time        |end_station_id|end_station_name            |bike_id|user_type    |model|source_year|
+--------+-------------+----------------+----------------+-----------------------------------+----------------+--------------+----------------------------+-------+-------------+-----+-----------+
|23529880|145          |08/01/2023 00:00|7101            |Lower Sherbourne St / The Esplanade|08/01/2023 00:02|7291          |190 Queens Quay E           |976    |Casual Member|NULL |2023       |
|23931602|909          |08/15/2023 09:28|7730            |NULL                               |08/15/2023 09:43|7719          |NULL                        |2738   |Casual Member|NULL |2023       |
|23529881|473       